# Políticas de acción aprendidas: método en la práctica

**Nivel:** 2  
**Requisitos previos:** Políticas de acción aprendidas nivel 1  
**Tiempo de lectura:** 25 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. El producto compara una FSM determinista y un árbol de comportamiento con puntajes logísticos y de árboles potenciados ajustados, una política recurrente y un bandido contextual. El resultado de la tarea es la poscondición independiente del software; coincidir con la demostración es otro objetivo.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/benchmark-adequacy.json`
- `data/derived/episodes/M03-C05-nominal-40000.json`
- `data/derived/episodes/M04-C05-nominal-40000.json`

## Método

Para un puntuador, el paquete publicado transforma una observación y sus candidatos actualmente admisibles en características por candidato, los puntúa y devuelve una decisión tipada. Los episodios congelados conservan la acción elegida, el puntaje, la identidad de política y el resultado final. Este notebook analiza esos registros sin copiar el algoritmo de puntuación.

Para el baseline logístico, \(p(a\mid x)=\sigma(w^T\phi(x,a))\); para árboles potenciados, el logit es una suma de puntajes de árboles. Estos modelos ordenan acciones registradas; ninguna ecuación concede autoridad ni equipara la coincidencia con una demostración al éxito verificado.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M03', 'M04']
sample_case = 'C05'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 2 | case: C05 | seed: 40000
Methods: M03, M04


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M03 | action= a-http_get | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-http_get', 'a-refresh', 'a-report', 'a-backoff', 'a-wait_dependency']
  success= 1 | steps= 3 | violations= 0
M04 | action= a-http_get | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-http_get', 'a-refresh', 'a-report', 'a-backoff', 'a-wait_dependency']
  success= 1 | steps= 3 | violations= 0


### Ejecuta las políticas publicadas y compara la corrida

In [4]:
# Los resultados de planificación se leen de las trazas congeladas; no hay llamada implícita a un LLM.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M03 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-http_get'}
M04 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-http_get'}


### Compara la secuencia completa y no solo la primera acción

In [5]:
# La igualdad de secuencia es diagnóstica; la verificación independiente decide el éxito.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M03 a-http_get -> a-backoff -> a-http_get | terminal= succeeded | verified= 1
M04 a-http_get -> a-backoff -> a-http_get | terminal= succeeded | verified= 1


## Resultados

Los checkpoints del paquete se publican como activos separados y no están versionados en este repositorio. Los resultados de los métodos ajustados son evaluaciones registradas y verificadas por hash, no modelos reajustados ahora. Una tasa en familias diseñadas no estima el rendimiento en tareas de software arbitrarias.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

Los puntuadores aprendidos se evalúan junto a baselines simples y sólidos. Algunos artefactos ajustados lograron éxito total en el núcleo, pero el techo del benchmark limita cualquier inferencia de superioridad.

## Referencias

- [XGB](https://arxiv.org/abs/1603.02754)
- [GRU](https://arxiv.org/abs/1406.1078)
- [LINUCB](https://arxiv.org/abs/1003.0146)